# Maps

## Agenda

- Diskussion: Vor- und Nachteile von array-basierten und verketteten Strukturen  
- Vergleich zu `set` und `dict`  
- Der **Map** ADT  
- Direkte Zugriffe via *Hashing*  
- Hashtabellen  
    - Kollisionen und das „Birthday-Problem“  
- Laufzeitanalyse & Diskussion

## Diskussion: Vor- und Nachteile von array-basierten und verketteten Strukturen

Zwischen dem array-basierten und dem verketteten Listen haben wir:

1. $O(1)$ Indexierung (array-basiert)  
2. $O(1)$ Anhängen (array-basiert & verkettet)  
3. $O(1)$ Einfügen/Löschen ohne Indexierung (verkettet)  
4. $O(N)$ lineare Suche (unsortiert)  
5. $O(\log N)$ binäre Suche, wenn sortiert (nur array-basierte Listen)

## Vergleich mit `set` und `dict`

Die Typen `set` und `dict` unterstützen keinen positionsbasierten Zugriff (d.h. per Index), aber sie unterstützen Nachschlagen/Suchen. Wie schnell sind sie im Vergleich zu Listen?

In [ ]:
def lin_search(lst, x):
    for y in lst:
        if x == y:
            return True
    else:
        return False
    
def bin_search(lst, x):
    # assumes lst is sorted
    low = 0
    hi  = len(lst)-1
    while low <= hi:
        mid = (low + hi) // 2
        if x < lst[mid]:
            hi  = mid - 1
        elif x < lst[mid]:
            low = mid + 1
        else:
            return True
    else:
        return False

In [ ]:
import timeit
import matplotlib.pyplot as plt
import numpy as np
import random

%matplotlib inline
plt.rcParams['figure.figsize'] = [10, 6] # set size of plot

ns = np.linspace(100, 10_000, 50, dtype=int)

ts_linsearch = [timeit.timeit('lin_search(lst, lst[-1])',
                              setup=f'lst = list(range({n}))',
                              globals=globals(),
                              number=100)
                for n in ns]

ts_binsearch = [timeit.timeit('bin_search(lst, 0)',
                              setup=f'lst = list(range({n}))',
                              globals=globals(),
                              number=100)
                for n in ns]

ts_setadd    = [timeit.timeit(f'st.add({n})',
                              setup=f'st = set(range({n}))',
                              globals=globals(),
                              number=100)
                for n in ns]


ts_setsearch = [timeit.timeit(f'{0} in st', # try for other values
                              setup=f'st = set(range({n}))',
                              globals=globals(),
                              number=100)
                for n in ns]

ts_dctadd    = [timeit.timeit(f'dct[{n}] = 0',
                              setup=f'dct = {{x:x for x in range({n})}}',
                              globals=globals(),
                              number=100)
                for n in ns]

ts_dctsearch = [timeit.timeit(f'{10} in dct', # try for other values
                              setup=f'dct = {{x:x for x in range({n})}}',
                              globals=globals(),
                              number=100)
                for n in ns]

In [ ]:
plt.plot(ns, ts_linsearch, 'sr')
plt.plot(ns, ts_binsearch, 'sg')
plt.plot(ns, ts_setadd, 'db')
plt.plot(ns, ts_setsearch, 'ob')
plt.plot(ns, ts_dctadd, 'dm');
plt.plot(ns, ts_dctsearch, 'om');
plt.legend(['Lin Search Lst','Bin Search Lst','Add Set', 'Search Set', 'Add Dict', 'Search Dict'])

Irgendwie scheint es, dass Mengen und Wörterbücher durch das Verwerfen von positionsbasiertem Zugriff und Manipulation Einfügen und Suchen in **konstanter Zeit** ermöglichen!

Wie ist dieser Zauber möglich?

## Der **Map** ADT

Wir werden uns zuerst auf den abstrakten Datentyp "*map*" (auch bekannt als "assoziatives Array" oder "Wörterbuch") konzentrieren, der verwendet wird, um Schlüssel (die eindeutig sein müssen) mit Werten zu verknüpfen.

Eine Map *legt* von sich aus keine Ordnung ihrer Inhalte fest — das heißt, eine Implementierung einer Map muss keinen positionsbasierten Zugriff auf Schlüssel unterstützen und auch keine konsistente Reihenfolge der Schlüssel gewährleisten.

Der Python-Typ `dict` ist eine Implementierung des Map-ADT.

### Aber wie sieht es mit Mengen (`Sets`) aus?

Hätten wir eine Implementierung einer Map, dann sollte es einfach sein, diese zur Implementierung eines Sets zu verwenden. Denke daran, dass eine „Menge“ ein mathematisches Objekt ist. Sie enthält jeden eindeutigen Wert nur einmal. D.h. es ist ein „Dict“, das nur Schlüssel ohne Werte enthält.

In [ ]:
class MySet:
    def __init__(self):
        self.dct = dict()
    
    def add(self, value):
        self.dct[value] = None
    
    def __contains__(self, value):
        return value in self.dct
        
    def intersection(self, other):
        assert isinstance(other, MySet)
        temp = MySet()
        for x in self:
            if x in other:
                temp.add(x)
        return temp

    
    def __iter__(self):
        return iter(self.dct)
            
    def __repr__(self):
        return '{' + ', '.join(repr(x) for x in self) + '}'

In [ ]:
st = MySet()

for c in 'hello world!':
    st.add(c)

st

In [ ]:
set('hello world!')

In [ ]:
st2 = MySet()

for c in 'farewell planet':
    st2.add(c)
    
st.intersection(st2)

In [ ]:
set('hello world!') & set('farewell planet')

### Eine einfache Map-Implementierung
Wir erstellen eine einfache "Map"-Implementierung unter Verwendung eines doppelt indizierten Python-Listenobjekts.

In [ ]:
class MapDS:
    def __init__(self):
        self.data = []
    
    def __setitem__(self, key, value):
        # searching if key is already in list
        for i in range(len(self.data)): #O(n)
            if self.data[i][0] == key:
                self.data[i][1] = value
        # didn't found the key so append a new key, value pair/list
        else:
            self.data.append([key,value])
       
    
    def __getitem__(self, key):
        for k,v in self.data: # O(n)
            if k == key:
                return v
        else:
            raise KeyError(key)

    def __contains__(self, key):
        for k,v in self.data: # O(n)
            if k == key:
                return True
        else:
            return False
        

In [ ]:
m = MapDS()
m['batman'] = 'bruce wayne'
m['superman'] = 'clark kent'
m['spiderman'] = 'peter parker'

In [ ]:
m['batman']

In [ ]:
m['batman'] = 'tony stark'

In [ ]:
m['batman']

Wie können wir den Sprung von linearer Laufzeitkomplexität zu konstanter schaffen?!

Am besten wäre es, einen Index zu haben. So würden wir beim Abrufen eines Schlüssels genau wissen, an welcher Position in der Liste unser Schlüssel gespeichert ist.

Also, wie können wir einen Index erstellen?

## Direkte Zugriffe über *Hashing*

Hashcodes (auch Hashwerte oder einfach Hashes genannt) sind einfache numerische Werte (meist Ganzzahlen), die für Objekte von Hashfunktionen berechnet werden. Ihre Implementierung basiert typischerweise auf einfachen mathematischen Operationen wie Multiplikation, Division und Modulo-Rechnung, die auf die Daten des Objekts angewendet werden.

Bauen wir uns selbst eine sehr einfache Hash-Funktion:

In [ ]:
# a simple hash function
def myhash(s):
    h = 0
    for c in s:
        h = 31 * h + ord(c)
    return h

myhash('hello')



Python bringt eine eigene (etwas komplexere) Hashfunktion mit:

In [ ]:
hash('hello')

In [ ]:
hash('batman')

In [ ]:
hash('batmen') 

In [ ]:
[hash(s) for s in ['different', 'objects', 'have', 'very', 'different', 'hashes']]

In [ ]:
[hash(s)%100 for s in ['different', 'objects', 'have', 'very', 'different', 'hashes']]

### Zufälliges Hashing

Die `hash`-Funktion in Python ist standardmäßig *randomisiert* – das heißt, jedes Mal, wenn ein Python-Interpreter gestartet wird, verwendet die Implementierung von `hash` einen anderen „Seed“ für den Zufallszahlengenerator, der zur Berechnung der Hashwerte verwendet wird. Während die für einen gegebenen Wert berechneten Hashcodes innerhalb einer Interpreter-Instanz konsistent sind, sind sie es nicht über verschiedene Instanzen hinweg! Das bedeutet, dass wir Hashcodes für Werte nicht auf der Festplatte speichern oder in einer Datenbank ablegen sollten, da Werte nach einem Neustart unserer Software mit großer Wahrscheinlichkeit unterschiedliche Hashcodes erhalten!

Warum macht Python das? Mehr dazu später!

## Hashtabellen

Eine **Hashtabelle** ist eine Implementierung des Map-ADT, die den Hashcode eines Schlüssels verwendet, um einen Index in einem Array zu berechnen, an dem das entsprechende Schlüssel/Wert-Paar gespeichert wird.
![](images/hash_table_buckets.png)

In [ ]:
class Hashtable:
    def __init__(self, n_buckets):
        self.buckets = [None] * n_buckets
        
    def __setitem__(self, key, value):
       bidx = hash(key) % len(self.buckets)
       self.buckets[bidx] = [key,value]
    
    
    def __getitem__(self, key):
        bidx = hash(key) % len(self.buckets)
        if self.buckets[bidx] is not None:
            return self.buckets[bidx][1]
        else:
            raise KeyError(key)
        
    def __contains__(self, key):
        try:
            _ = self[key]
            return True
        except:
            return False

In [ ]:
ht = Hashtable(100)
ht['spiderman'] = 'peter parker'
ht['batman'] = 'bruce wayne'
ht['superman'] = 'clark kent'

In [ ]:
ht['spiderman']

In [ ]:
ht['batman']

In [ ]:
ht['superman']

Und nun ist der Aufwand, einen Schlüssel zu finden, nicht mehr $O(N)$, sondern hängt von der Hash-Funktion ab.

## Über Kollisionen

### Das „Geburtstagsproblem“

**Problemstellung:**  
Gegeben sind $n$ Personen auf einer Party. Wie wahrscheinlich ist es, dass mindestens zwei Personen am selben Tag Geburtstag haben? (Wir gehen von einem festen Jahr mit 365 Tagen aus.)

Wie viele Personen braucht man deiner Meinung nach, damit die Wahrscheinlichkeit für einen gemeinsamen Geburtstag bei 50% liegt? Und bei 99%?

Der Trick zur Lösung des Geburtstagsproblems ist, das Komplement zu betrachten, also das Ereignis „kein geteilter Geburtstag“. Die Wahrscheinlichkeit, dass mindestens zwei Personen denselben Geburtstag haben, ist:

$$
1 - P(\text{keine gemeinsamen Geburtstage})
$$

Für 1 Person liegt die Wahrscheinlichkeit bei:
$$
P_1 = 1
$$

Für 2 Personen:
$$
P_2 = \frac{365}{365} \cdot \frac{364}{365}
$$

Für 3 Personen:
$$
P_3 = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365}
$$

Für 4 Personen:
$$
P_4 = \frac{365}{365} \cdot \frac{364}{365} \cdot \frac{363}{365} \cdot \frac{362}{365}
$$

Für $n$ Personen:
$$
P_n = \prod_{k=0}^{n-1} \frac{365 - k}{365}
$$

### Kollisionswahrscheinlichkeiten

In [ ]:
def birthday_p(n_people):
    p_inv = 1
    for n in range(365, 365-n_people, -1):
        p_inv *= n / 365
    return 1 - p_inv

In [ ]:
birthday_p(35)

In [ ]:
1-364/365*363/365

In [ ]:
n_people = range(1, 80)
plt.plot(n_people, [birthday_p(n) for n in n_people]);
plt.title("Birthday Paradox");
plt.xlabel("Number of People");
plt.ylabel("Probability of shared birthday");
plt.annotate("More than 50% chance at 23 people", xy=(23, 0.5), xytext=(25, 0.3),
             arrowprops=dict(arrowstyle="->", color='black'));
plt.annotate("More than 80% chance at 30 people", xy=(35, 0.8), xytext=(40, 0.6),
             arrowprops=dict(arrowstyle="->", color='black'));

### Allgemeine Kollisionsstatistiken

Wiederhole das Geburtstagsproblem, aber mit einer gegebenen Anzahl von Werten und „Buckets“, die ihnen zugewiesen sind. Wie wahrscheinlich ist es, dass zwei oder mehr Werte auf denselben Bucket abgebildet werden?

In [ ]:
def collision_p(n_values, n_buckets):
    p_inv = 1
    for n in range(n_buckets, n_buckets-n_values, -1):
        p_inv *= n / n_buckets
    return 1 - p_inv

In [ ]:
collision_p(23, 365) # same as birthday problem, for 23 people

In [ ]:
collision_p(10, 100)

In [ ]:
collision_p(100, 1000)

In [ ]:
# keeping number of values fixed at 100, but vary number of buckets: visualize probability of collision
n_buckets = range(100, 100001, 1000)
plt.plot(n_buckets, [collision_p(100, nb) for nb in n_buckets]);

## Umgang mit Kollisionen

Um Kollisionen in einer Hashtabelle zu behandeln, erstellen wir einfach eine "Kette" von Schlüssel/Wert-Paaren für jeden Bucket, in dem Kollisionen auftreten. Die Kette muss eine Datenstruktur sein, die eine schnelle Einfügung unterstützt – die natürliche Wahl: die verkettete Liste!

![](images/hash_table_collision.png)

In [ ]:
class Hashtable:
    class Node:
        def __init__(self, key, val, next=None):
            self.key = key
            self.val = val
            self.next = next
            
    def __init__(self, n_buckets=1000):
        self.buckets = [None] * n_buckets
        
    def __setitem__(self, key, val):
        bidx = hash(key) % len(self.buckets)
        if self.buckets[bidx] is None:
            self.buckets[bidx] = Hashtable.Node(key, val, None)
        else:
            if self.buckets[bidx].key == key:
                self.buckets[bidx].val = val
            else:
                newNode = Hashtable.Node(key, val, self.buckets[bidx])
                self.buckets[bidx] = newNode

    
    def __getitem__(self, key):
        bidx = hash(key) % len(self.buckets)
        if self.buckets[bidx] is None:
            raise KeyError(key)
        else:
            head = self.buckets[bidx]
            while head:             # O (n)
                if head.key == key:
                    return head.val
                else:
                    head = head.next
            else:
                raise KeyError(key)

    
    def __contains__(self, key):
        try:
            _ = self[key]
            return True
        except:
            return False

In [ ]:
ht = Hashtable(100)
ht['batman'] = 'bruce wayne'
ht['superman'] = 'clark kent'
ht['spiderman'] = 'peter parker'
ht['ironman'] = 'tony stark'

In [ ]:
ht['batman']

In [ ]:
ht['superman']

In [ ]:
ht['spiderman']

In [ ]:
def init_ht(size):
    ht = Hashtable(size)
    for x in range(size):
        ht[x] = x
    return ht

ns = np.linspace(100, 10_000, 50, dtype=int)
ts_htsearch = [timeit.timeit(f'{0} in ht',
                             setup='ht = init_ht({})'.format(n),
                             globals=globals(),
                             number=100)
               for n in ns]

In [ ]:
plt.plot(ns, ts_binsearch, 'ro')
plt.plot(ns, ts_htsearch, 'gs')
plt.plot(ns, ts_dctsearch, 'b^');
plt.legend(['Binary Search', 'Hashtable Search', 'Dict Search'])

## Lose Enden

### Iteration

In [ ]:
class Hashtable(Hashtable):
    def __iter__(self):
        for i in range(len(self.buckets)):
            if self.buckets[i] is not None:
                head = self.buckets[i]
                while head:
                    yield head.key
                    head = head.next

In [ ]:
ht = Hashtable(100)
ht['batman'] = 'bruce wayne'
ht['superman'] = 'clark kent'
ht['spiderman'] = 'peter parker'

In [ ]:
for k in ht:
    print(k)

### Schlüsselreihenfolge

In [ ]:
ht = Hashtable()
d = {}
for x in 'apple banana cat dog elephant'.split():
    d[x[0]] = x
    ht[x[0]] = x

In [ ]:
for k in d:
    print(k, '=>', d[k])

In [ ]:
for k in ht:
    print(k, '=>', ht[k])

Da unsere Hashtable von der Funktion ```hash(key) % numberOfBuckets``` abhängt, hängt die Reihenfolge der Objekte von den berechneten Hash-Werten ab.

### Load-Faktor & Rehashing

Es ist klar, dass das Verhältnis der Anzahl der Schlüssel zur Anzahl der Buckets (bekannt als **Load-Faktor**) einen erheblichen Einfluss auf die Leistung einer Hashtabelle haben kann.

Eine feste Anzahl von Buckets macht keinen Sinn, da sie bei einer kleinen Anzahl von Schlüsseln verschwenderisch sein kann und bei einer relativ großen Anzahl von Schlüsseln schlecht skaliert. Ebenso macht es keinen Sinn, dass der Benutzer der Hashtabelle manuell die Anzahl der Buckets festlegt (was ein niedrigstufiges Implementierungsdetail ist).

Stattdessen würde eine praktische Hashtabellen-Implementierung mit einer relativ kleinen Anzahl von Buckets starten, und wenn/sofern der Load-Faktor einen bestimmten Schwellenwert (typischerweise 1) überschreitet, *erhöht sie dynamisch die Anzahl der Buckets* (typischerweise auf das Doppelte der vorherigen Anzahl). Dies erfordert, dass alle vorhandenen Schlüssel *neu gehasht* werden, um auf die neuen Buckets verteilt zu werden (warum?).

Wie ist die Laufzeitkomplexität des Rehashings einer Hashtabelle mit *N* Schlüsseln?

### Uniformes Hashing

Letztendlich hängt die Leistung einer Hashtabelle auch stark davon ab, dass die Hashcodes *gleichmäßig verteilt* sind — das heißt, statistisch gesehen hat jeder Bucket ungefähr die gleiche Anzahl von Schlüsseln, die darauf abgebildet werden. Das Entwerfen von Hashfunktionen, die dies gewährleisten, ist ein algorithmisches Problem, das außerhalb des Umfangs dieses Kurses liegt!

## Laufzeitanalyse & Diskussion

Für eine Hashtabelle mit $N$ Schlüssel/Wert-Einträgen gilt im schlimmstmöglichen Szenario, dass alle Schlüssel auf denselben Bucket abgebildet werden, und wir am Ende eine Kette von $N$ Kollisionen durchlaufen müssen beim Einfügen/Suchen/Löschen, was uns die folgenden *Laufzeitkomplexitäten im schlimmsten Fall* ergibt:

- Einfügen: $O(N)$
- Suchen: $O(N)$
- Löschen: $O(N)$

ABER, wenn wir uniformes Hashing und das oben beschriebene Rehashing-Verhalten annehmen, ist es möglich zu beweisen, dass Hashtabellen eine *amortisierte (d.h. durchschnittliche) Laufzeitkomplexität* von $O(1)$ haben. Der Beweis dafür liegt ebenfalls außerhalb des Umfangs dieses Kurses (wird jedoch durch empirische Daten bestätigt).

### Denial-of-Service-Angriffe und zufälliges Hashing

Hashtables sind einzigartig, da sie auf einer guten durchschnittlichen Laufzeitkomplexität basieren und auf der unglaublich geringen Wahrscheinlichkeit, dass eine große Anzahl von Kollisionen auftritt.

Aber was, wenn jemand im Voraus wüsste, welche Werte auf welche Hashes abgebildet werden, und absichtlich eine große Anzahl von Schlüsseln in eine Hashtable eingibt, die zu Kollisionen führen?

Dies ist die Grundlage eines **Denial-of-Service-Angriffs**, der darauf abzielt, die schlimmste Laufzeitkomplexität einer Hashtable auszunutzen! Die gute Nachricht: Python macht dies durch die Standard-Randomisierung von Hashes deutlich schwieriger – aber es bedeutet auch, dass Sie vorsichtig sein müssen, Ihre Hashfunktion zu schützen, wenn Sie selbst eine implementieren.

## Vokabelliste

- Hashtabelle  
- Hashing und Hashes  
- Kollision  
- Hash-Buckets & Ketten  
- Geburtstagsproblem  
- Auslastungsfaktor  
- Rehashing  
- Denial-of-Service-Angriff

## Nachtrag: Zur *Hashbarkeit*

Erinnere dich: *Ein gegebenes Objekt muss immer auf denselben Wert gehasht werden*. Dies ist erforderlich, damit wir das Objekt stets demselben Hash-Bucket zuordnen können.

Hashcodes für Sammlungen von Objekten werden üblicherweise aus den Hashcodes ihrer Inhalte berechnet, z.B. ist der Hash eines Tupels eine Funktion der Hashes der Objekte in diesem Tupel:

In [ ]:
hash(('two', 'strings'))

Dies ist nützlich. Es erlaubt uns beispielsweise, ein Tupel als Schlüssel für eine Hashtabelle zu verwenden.

Wenn die Sammlung von Objekten jedoch *veränderlich* ist — d.h., wir können ihren Inhalt ändern — bedeutet dies, dass wir potenziell ihren Hashcode ändern können.

Wenn wir eine solche Sammlung als Schlüssel in einer Hashtabelle verwenden und die Sammlung nach der Zuordnung zu einem bestimmten Bucket ändern, führt dies zu einem ernsthaften Problem: Die Sammlung befindet sich möglicherweise nun im falschen Bucket (da sie basierend auf ihrem ursprünglichen Hashcode einem Bucket zugewiesen wurde)!

Aus diesem Grund sind standardmäßig nur unveränderliche Typen in Python hashbar. Während wir also Ganzzahlen, Zeichenketten und Tupel als Schlüssel in Dictionaries verwenden können, können Listen (die veränderlich sind) nicht verwendet werden. Tatsächlich markiert Python eingebaute veränderliche Typen als „unhashable“, z.B.

In [ ]:
hash([1, 2, 3])

Das gesagt, unterstützt Python das Hashing von Instanzen benutzerdefinierter Klassen (die veränderlich sind). Dies liegt daran, dass die Standardimplementierung der Hash-Funktion nicht auf dem Inhalt von Instanzen benutzerdefinierter Klassen basiert. Zum Beispiel,

In [ ]:
class Student:
    def __init__(self, fname, lname):
        self.fname = fname
        self.lname = lname

In [ ]:
s = Student('John', 'Doe')
hash(s)

In [ ]:
s.fname = 'Jane'
hash(s) # same as before mutation

Wir können das Standardverhalten ändern, indem wir unsere eigene Hash-Funktion in `__hash__` bereitstellen, z.B.,

In [ ]:
class Student:
    def __init__(self, fname, lname):
        self.fname = fname
        self.lname = lname
        
    def __hash__(self):
        return hash(self.fname) + hash(self.lname)

In [ ]:
s = Student('John', 'Doe')
hash(s)

In [ ]:
s.fname = 'Jane'
hash(s)

Aber Vorsicht: Instanzen dieser Klasse sind nicht mehr geeignet, als Schlüssel in Hashtabellen (oder Wörterbüchern) verwendet zu werden, wenn Sie vorhaben, sie nach der Verwendung als Schlüssel zu verändern!